# Embryo workflow: independent stages

Run automatic segmentation, training, classification, neighbour counting, or plotting as independent tasks. First run the environment and configuration cells, then set the required `RUN_...` switch to `True` and execute that task's cell. All tasks are initially disabled.

Inputs are read only. Results default to `outputs/staged_notebook/`. See the [main README](../README.md) for setup and terminal commands.

In [ ]:
from pathlib import Path
import json
import sys

working = Path.cwd().resolve()
ROOT = next((path for path in [working, *working.parents]
             if (path / 'EmbryoAnalyser/workflow.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start the notebook from the project root or EmbryoAnalyser directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display, Image
from EmbryoAnalyser.workflow import DEFAULT_MODEL_FILE, train_workflow, classify_workflow, distribution_workflow
from EmbryoAnalyser.neighbours import run_neighbours

print('Project:', ROOT)

from EmbryoAnalyser.segmentation import segmentation_workflow
from EmbryoAnalyser.measurements import measurement_workflow


## Select tasks and inputs

`SEGMENT_INPUTS` accepts one TIFF, multiple TIFFs, or a recursive directory. Segmentation reuses the separate Cellpose environment and `cpsam_v2`; inference is automatic and does not open a GUI.

For raw Z-stacks, explicitly set `Z_PROJECTION = 'max'` for per-channel maximum-intensity projection, or set a zero-based `Z_PLANE`. Two-dimensional TIFFs remain unchanged. Do not select both options. The existing neighbour macro uses 2D ROIs.

Classification still needs one measurement CSV per embryo. ROI ZIPs do not replace those CSVs. A saved model can be used directly without rerunning training.

In [ ]:
OUTPUT = ROOT / 'outputs/staged_notebook'
SEGMENT_INPUTS = [ROOT / 'dataset/fixed_EM/processed/del15/s9_1/Composite.tif']
CSV_INPUTS = [ROOT / 'dataset/raw_dataset/E-CadGFP']
TRAIN_INPUTS = [ROOT / 'dataset/raw_dataset/gap43-mCherry']
PREPROCESS_IMAGE_INPUTS = [ROOT / 'dataset/fixed_EM/processed/del15/s9_1/Composite.tif']
NEIGHBOUR_INPUTS = [ROOT / 'dataset/fixed_EM/processed/del15/s9_1/Composite.tif']
EXISTING_COUNTS = [ROOT / 'dataset/fixed_EM/processed/del15/s9_1/slow_neighbour_counting.csv']
MODEL_FILE = DEFAULT_MODEL_FILE
MODELS = 'both'  # 'rf' / 'svm' / 'both'; applies when training
CELLPOSE_PYTHON = None  # default: existing cellpose_env; can set full Python path
Z_PROJECTION = None  # for raw Z-stack use 'max', or set Z_PLANE
Z_PLANE = None  # zero-based plane index
DIAMETER = None  # keep the original Cellpose default
USE_GPU = False
SEGMENT_TIMEOUT = None
FIJI_TIMEOUT = None
RUN_SEGMENTATION = False
RUN_MEASUREMENT = False
RUN_MASKS = False
RUN_TRAINING = False
RUN_CLASSIFICATION = False
RUN_NEIGHBOURS = False
RUN_DISTRIBUTION = False


## 1. Automatic segmentation

Enable `RUN_SEGMENTATION=True` and run this cell. It displays each image's ROI count and sets `NEIGHBOUR_INPUTS` to the prepared images directory for the current session. No other task starts automatically.

In a later session, set `NEIGHBOUR_INPUTS` to the report's saved `image_dir` to count neighbours without repeating segmentation.

In [ ]:
if RUN_SEGMENTATION:
    segmented = segmentation_workflow(
        SEGMENT_INPUTS, OUTPUT / 'segmentation', python_executable=CELLPOSE_PYTHON,
        z_projection=Z_PROJECTION, z_plane=Z_PLANE, diameter=DIAMETER,
        use_gpu=USE_GPU, timeout=SEGMENT_TIMEOUT)
    display(pd.DataFrame(segmented['samples'])[['input_image', 'n_rois', 'image_file', 'roi_file']])
    NEIGHBOUR_INPUTS = [Path(segmented['image_dir'])]
    print('Images directory for neighbour counting:', segmented['image_dir'])
    print('Segmentation report:', segmented['report_json'])
    PREPROCESS_IMAGE_INPUTS = [Path(segmented['image_dir'])]


## 2. Image measurements and masks

Set `PREPROCESS_IMAGE_INPUTS` to TIFF images with matching ROI ZIPs. Enable `RUN_MEASUREMENT` for raw measurement CSVs and/or `RUN_MASKS` for PNG masks. All outputs go to a new folder under `OUTPUT / preprocessing`. Measurement updates `CSV_INPUTS` for later training/classification. CSV cleaning, feature binning, and scaling run automatically within those tasks; do not replace raw CSVs with binned profiles.


In [ ]:
if RUN_MEASUREMENT or RUN_MASKS:
    prepared = measurement_workflow(PREPROCESS_IMAGE_INPUTS, OUTPUT / 'preprocessing',
                                    measure=RUN_MEASUREMENT, export_masks=RUN_MASKS,
                                    timeout=FIJI_TIMEOUT)
    if RUN_MEASUREMENT:
        CSV_INPUTS = [Path(prepared['csv_dir'])]
        TRAIN_INPUTS = CSV_INPUTS
    display(pd.DataFrame(prepared['samples']))
    print('Prepared inputs:', prepared['run_dir'])


## 3. RF / SVM training (optional)

Classification already uses the bundled RF/SVM model trained on all 14 gap43 embryos. Enable this task only to train a custom model; it updates `MODEL_FILE` for the current session.

In [ ]:
if RUN_TRAINING:
    trained = train_workflow(TRAIN_INPUTS, OUTPUT / 'training', model=MODELS)
    MODEL_FILE = trained['model_file']
    print('Model:', MODEL_FILE)


## 4. CSV classification

Enable `RUN_CLASSIFICATION=True`. The task uses the models and preprocessing state in `MODEL_FILE`; it does not start segmentation or Fiji.

In [ ]:
if RUN_CLASSIFICATION:
    classified = classify_workflow(CSV_INPUTS, MODEL_FILE, OUTPUT / 'classification')
    display(classified['predictions'])
    print('Classification CSV:', classified['predictions_csv'])


## 5. Count neighbours and display distributions

Enable `RUN_NEIGHBOURS=True`. `NEIGHBOUR_INPUTS` must refer to images with matching ROI ZIPs or their directory. The original Fiji macro runs on copies, followed by Python distribution tables and plots.

In [ ]:
if RUN_NEIGHBOURS:
    counted = run_neighbours(NEIGHBOUR_INPUTS, OUTPUT / 'neighbours', timeout=FIJI_TIMEOUT)
    display(pd.read_csv(counted['summary_csv']))
    for sample in counted['samples']:
        display(pd.read_csv(sample['distribution_csv']))
        display(Image(filename=sample['png']))
    print('Neighbour summary:', counted['summary_csv'])


## 6. Plot existing counts only

Enable `RUN_DISTRIBUTION=True`. Existing `slow_neighbour_counting.csv` inputs are plotted without segmentation or Fiji.

In [ ]:
if RUN_DISTRIBUTION:
    plotted = distribution_workflow(EXISTING_COUNTS, OUTPUT / 'distributions')
    for sample in plotted['samples']:
        display(pd.read_csv(sample['distribution_csv']))
        display(Image(filename=sample['png']))
    print('Distribution summary:', plotted['summary_csv'])
